In [1]:
try:
    import google.colab  # noqa: F401

    # specify the version of DataEval (==X.XX.X) for versions other than the latest
    %pip install -q dataeval
except Exception:
    pass

In [2]:
import json
import warnings
from pathlib import Path
from tempfile import mkdtemp

import numpy as np

from dataeval import Metadata
from dataeval.bias import Balance

In [3]:


def collection(seed: int, mean_temp: float, n: int = 600):
    rng = np.random.default_rng(seed)
    temp = rng.normal(mean_temp, 9.0, n)
    factors = {
        "temp_c": temp,
        "haze": np.clip(0.55 * (temp - mean_temp) / 9.0 + rng.normal(0.0, 0.9, n), -3, 3),
        "site": np.array(["ridge", "valley", "coast"])[rng.integers(0, 3, n)],
    }
    return factors, rng.integers(0, 4, n)


winter_factors, winter_labels = collection(seed=0, mean_temp=-2.0)
spring_factors, spring_labels = collection(seed=1, mean_temp=14.0)

In [4]:
FREEZING = [-np.inf, 0.0, 10.0, np.inf]  # below freezing / cold / mild
SITES = ["ridge", "valley", "coast"]

winter = Metadata.from_factors(
    winter_factors,
    class_labels=winter_labels,
    continuous_factor_bins={"temp_c": FREEZING},
    factor_levels={"site": SITES},
)

for name, spec in winter.encoding().items():
    print(f"{name:8} {type(spec).__name__:10} provenance={spec.provenance}")

haze     BinSpec    provenance=derived
site     LevelSpec  provenance=declared
temp_c   BinSpec    provenance=edges


/tmp/ipykernel_4775/1036466439.py:11: UserWarning: `haze` was binned automatically ('uniform_width') because no bins were declared. The bin count is derived from the data, so it is not stable across samples and the same factor measured twice may not be comparable. Declare cutoffs with continuous_factor_bins={"haze": [...]} to control this.
  for name, spec in winter.encoding().items():


In [5]:
try:
    incomplete = Metadata.from_factors(
        winter_factors,
        class_labels=winter_labels,
        factor_levels={"site": ["ridge", "valley"]},
        strict=True,
    )
    _ = incomplete.factor_data
except ValueError as error:
    print(error)

Values ['coast'] are not in this factor's declared vocabulary ['ridge', 'valley']. Pass strict=False to admit them, or widen the declaration.


In [6]:
before = winter.factor_data.copy()
winter.accept()
print(f"provenance: { ({n: s.provenance for n, s in winter.encoding().items()}) }")
print(f"codes unchanged: {np.array_equal(before, winter.factor_data)}")

provenance: {'haze': 'accepted', 'site': 'declared', 'temp_c': 'edges'}
codes unchanged: True


In [7]:
workspace = Path(mkdtemp(prefix="dataeval-encoding-"))
descriptor = workspace / "encoding.json"
winter.export_encoding(descriptor)

print(json.dumps(json.loads(descriptor.read_text())["factors"]["temp_c"], indent=2))

{
  "edges": [
    "-inf",
    0.0,
    10.0,
    "inf"
  ],
  "kind": "bins",
  "method": null,
  "provenance": "edges"
}


In [8]:
archive = workspace / "winter.dem"
winter.save(archive)
restored = Metadata.load(archive)

print(f"edges:  {restored.encoding('temp_c').edges}")
print(f"digest: {restored.encoding_digest == winter.encoding_digest}")

edges:  (-inf, 0.0, 10.0, inf)
digest: True


In [9]:
spring = Metadata.from_factors(spring_factors, class_labels=spring_labels, encoding=descriptor)

print(f"winter temp_c: {winter.encoding('temp_c').edges}")
print(f"spring temp_c: {spring.encoding('temp_c').edges}")

winter temp_c: (-inf, 0.0, 10.0, inf)
spring temp_c: (-inf, 0.0, 10.0, inf)


In [10]:
print(f"winter: {winter.encoding_digest}")
print(f"spring: {spring.encoding_digest}")
print(f"comparable: {winter.encoding_digest == spring.encoding_digest}")

result = Balance().evaluate(spring)
print(f"on the result: {result.meta().state['encoding_digest']}")

winter: 619615eaeb7391db
spring: 619615eaeb7391db
comparable: True
on the result: 619615eaeb7391db


In [11]:
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    stale = Metadata.from_factors(
        spring_factors,
        class_labels=spring_labels,
        continuous_factor_bins={"temp_c": [-np.inf, -20.0, -10.0, 0.0, np.inf]},
    )
    _ = stale.factor_data

for warning in caught:
    if "bins unused" in str(warning.message):
        print(str(warning.message))

Declared cuts left bins unused: temp_c (3 of 4 bins hold rows). The encoding still applies and the codes are unchanged; this is the data no longer matching the policy, not an error. Call reencode() to refit — which moves codes — or leave it and read the empty groups as the finding they are.


In [12]:
# TEST ASSERTION CELL ###
cold = Metadata.from_factors(winter_factors, class_labels=winter_labels)
warm = Metadata.from_factors(spring_factors, class_labels=spring_labels)
assert cold.encoding("temp_c").provenance == "derived"
assert not np.array_equal(cold.encoding("temp_c").edges, warm.encoding("temp_c").edges)

assert {name: spec.provenance for name, spec in winter.encoding().items()} == {
    "haze": "accepted",
    "site": "declared",
    "temp_c": "edges",
}
try:
    strict = Metadata.from_factors(
        winter_factors, class_labels=winter_labels, factor_levels={"site": ["ridge", "valley"]}, strict=True
    )
    _ = strict.factor_data
except ValueError:
    pass
else:
    raise AssertionError("a value outside a strict vocabulary should be refused")

assert json.loads(descriptor.read_text())["factors"]["temp_c"]["edges"][0] == "-inf"
assert np.array_equal(restored.encoding("temp_c").edges, winter.encoding("temp_c").edges)
assert restored.encoding_digest == winter.encoding_digest
assert np.array_equal(spring.encoding("temp_c").edges, winter.encoding("temp_c").edges)
assert spring.encoding_digest == winter.encoding_digest
assert result.meta().state["encoding_digest"] == spring.encoding_digest
assert any("bins unused" in str(warning.message) for warning in caught)

/tmp/ipykernel_4775/1420384198.py:4: UserWarning: `haze` and `temp_c` were binned automatically ('uniform_width') because no bins were declared. The bin count is derived from the data, so it is not stable across samples and the same factor measured twice may not be comparable. Declare cutoffs with continuous_factor_bins={"haze": [...]} to control this.
  assert cold.encoding("temp_c").provenance == "derived"
/tmp/ipykernel_4775/1420384198.py:5: UserWarning: `haze` and `temp_c` were binned automatically ('uniform_width') because no bins were declared. The bin count is derived from the data, so it is not stable across samples and the same factor measured twice may not be comparable. Declare cutoffs with continuous_factor_bins={"haze": [...]} to control this.
  assert not np.array_equal(cold.encoding("temp_c").edges, warm.encoding("temp_c").edges)
